# 12 — Single and multiple shooting

**Objectives.** Derive both transcriptions and expose the sparse continuity constraints. For $x_{k+1}=F(x_k,u_k)$, single shooting eliminates every state by repeated substitution. Multiple shooting keeps $X_k$ as decisions and imposes $X_{k+1}-F(X_k,U_k)=0$. The latter is larger but sparse and permits state initialization and path constraints directly.


In [ ]:
import casadi as ca
import numpy as np
from casadi_tutorial.integration import rk4_step
N, horizon = 20, 2.0; dt = horizon/N
def rhs(x,u): return ca.vertcat(x[1], u[0])
x_sym=ca.MX.sym('x',2); u_sym=ca.MX.sym('u',1)
F=ca.Function('F',[x_sym,u_sym],[rk4_step(rhs,x_sym,u_sym,dt)])
opti=ca.Opti(); X=opti.variable(2,N+1); U=opti.variable(1,N)
opti.subject_to(X[:,0]==[0,0])
for k in range(N): opti.subject_to(X[:,k+1]==F(X[:,k],U[:,k]))
opti.subject_to(X[:,-1]==[1,0]); opti.subject_to(opti.bounded(-2,U,2))
opti.minimize(dt*ca.sumsqr(U)); opti.set_initial(X[0,:],np.linspace(0,1,N+1))
opti.solver('ipopt',{'print_time':False},{'print_level':0})
sol=opti.solve(); print('objective',sol.value(opti.f),'iterations',sol.stats()['iter_count'])
assert np.max(np.abs(sol.value(X[:,-1])-np.array([1,0]))) < 1e-7


In [ ]:
decision = ca.vertcat(ca.vec(X), ca.vec(U))
constraint_jacobian = ca.jacobian(opti.g, decision)
print('constraint Jacobian:', constraint_jacobian.shape, 'nnz:', constraint_jacobian.nnz())


Single shooting is easy and has few variables but can be ill-conditioned for unstable/long horizons. Multiple shooting has a block-banded Jacobian and usually offers better initialization. Mesh refinement compares solutions at increasing $N$; it is not enough that one mesh converges.

**Recap:** transcription determines NLP structure before the solver sees the problem.
